# Building a LangChain Chain to Summarize Text

In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from tavily import TavilyClient

load_dotenv()

tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [ ]:
from typing import Dict, Any

@tool(description="Search the web for information")
def web_search(query: str) -> Dict[str, Any]:
    return tavily_client.search(query)

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("ollama:gpt-oss:latest", temperature=0.15)

agent = create_agent(
    model=model,
    tools=[web_search],
    checkpointer=InMemorySaver(),
)


In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="""given the information about  Elon Reeve Musk, I want you to create:
    1. A short summary
    2. two interesting facts about them""")
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [question]},
    config,  
)

In [ ]:
last_message = response['messages'][-1]
print(last_message.content)
print(json.dumps(last_message.usage_metadata, indent=4))
print(json.dumps(last_message.response_metadata, indent=4))